# Week 5 Lab — Introduction to pandas

**HWRS 564a · Fall 2026**

Everything you did last week with numpy works on numbers in a grid. Real data
comes with **labels** — this column is the well depth, that row is site
320906110592701 — and keeping track of which column was index 3 is how mistakes
happen.

`pandas` is numpy with labels on it. This week we use it on the real thing:
every USGS monitoring well in the Tucson basin, and a century of water levels
from the best-monitored of them.

## How to use this notebook

Run each cell with **Shift+Enter**. Cells marked  **`# YOUR TURN`**  have
something for you to write. Cells marked **`# CHECK`** verify your answer — if
they run without complaint, you're right.

> **Before you submit anything all semester:** *Kernel → Restart Kernel and Run
> All Cells*. A notebook that only works when run out of order is not finished.


## Learning objectives

By the end of this notebook you can:

1. Read a CSV into a `DataFrame` and inspect it before trusting it
2. Select columns, rows, and both at once with `.loc` and `.iloc`
3. Filter rows with a boolean mask, including several conditions at once
4. Compute a new column from existing ones
5. Read an Excel file, because collaborators will send you them forever
6. Make a first plot straight from a DataFrame

---

## Part 1 — Reading data, and looking before you leap

Every dataset in this course lives in `data/`, referenced by a path relative to
the repo root. Never a hardcoded `/Users/...` — your notebook has to run in
someone else's codespace.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Walk up from wherever this notebook is until we find the repo root, then
# point at data/. Counting "../.." works right up until you move the file.
ROOT = next(p for p in Path.cwd().parents if (p / "pyproject.toml").exists())
DATA = ROOT / "data"
print(f"reading from {DATA}")
print(sorted(p.name for p in DATA.glob("*")))

In [ ]:
wells = pd.read_csv(DATA / "tucson_basin_wells.csv", dtype={"site_no": str})

print(f"{len(wells):,} wells, {wells.shape[1]} columns")
wells.head()

`dtype={"site_no": str}` is not decoration. Site numbers are 15-digit
identifiers, and pandas would happily read them as integers and drop leading
zeros — quietly breaking every join you make later.

**Four things to check before you trust a table.** Do these every time.

In [ ]:
print("--- dtypes: is anything a string that should be a number? ---")
print(wells.dtypes)

In [ ]:
print("--- how much is missing? ---")
print(wells.isna().sum())

In [ ]:
print("--- do the numbers look physical? ---")
wells[["land_surface_elev_ft", "well_depth_ft", "latitude", "longitude"]].describe()

Read that table. The wells range from 30 ft to over 3,000 ft deep, land surface
runs 1,900–3,600 ft, and `aquifer_code` is missing for almost every well. None
of that is wrong, but all of it is worth knowing before you compute a mean.

### YOUR TURN 1

Answer three questions about the table, in code rather than by reading the
output above:

- `n_wells` — how many wells are there?
- `n_missing_depth` — how many are missing `well_depth_ft`?
- `deepest_ft` — how deep is the deepest well?

In [ ]:
# YOUR TURN
n_wells = ...
n_missing_depth = ...
deepest_ft = ...

In [ ]:
# CHECK
assert n_wells == len(wells), f"expected {len(wells)}, got {n_wells}"
assert n_missing_depth == wells["well_depth_ft"].isna().sum()
assert abs(deepest_ft - wells["well_depth_ft"].max()) < 1e-9
print(f"{n_wells:,} wells; {n_missing_depth} missing a depth; "
      f"deepest is {deepest_ft:,.0f} ft.")
print("Correct.")

---

## Part 2 — Getting at the data

A `DataFrame` is a dictionary of columns. One column is a `Series`.

In [ ]:
depths = wells["well_depth_ft"]

print(type(depths))
print(depths.head(3))
print(f"\nmean depth: {depths.mean():.0f} ft   (NaNs are skipped, silently)")

A **list** of column names gives you back a DataFrame with those columns:

In [ ]:
wells[["site_no", "station_name", "well_depth_ft"]].head(3)

For rows, there are two accessors and the distinction matters:

- **`.loc[]`** works on **labels** — the index value, and the column name
- **`.iloc[]`** works on **integer positions** — like a numpy array

They agree here only because the index happens to be 0, 1, 2, ...

In [ ]:
print(".iloc[3] — the fourth row by position")
print(wells.iloc[3][["site_no", "well_depth_ft"]])
print()
print(".loc[3] — the row whose index label is 3")
print(wells.loc[3][["site_no", "well_depth_ft"]])

Set a meaningful index and they part company immediately:

In [ ]:
by_site = wells.set_index("site_no")

print(by_site.loc["320906110592701", ["station_name", "well_depth_ft"]])
print()
print("...and .iloc[0] is still just 'whatever is first':")
print(by_site.iloc[0][["station_name", "well_depth_ft"]])

`.loc` takes rows **and** columns: `.loc[rows, columns]`. A slice with `.loc` is
**inclusive** of its endpoint, unlike every other slice in Python. That
inconsistency is real and it catches everyone once.

In [ ]:
print("iloc[2:5] gives 3 rows:", len(wells.iloc[2:5]))
print("loc[2:5]  gives 4 rows:", len(wells.loc[2:5]))

### YOUR TURN 2

Using `.loc`, pull out the `station_name` and `land_surface_elev_ft` of the well
with site number `"320824110593001"` from `by_site`, into a Series called
`one_well`.

In [ ]:
# YOUR TURN
one_well = ...

In [ ]:
# CHECK
assert isinstance(one_well, pd.Series), f"expected a Series, got {type(one_well)}"
assert list(one_well.index) == ["station_name", "land_surface_elev_ft"], \
    f"wrong columns: {list(one_well.index)}"
assert one_well["station_name"] == "D-15-13 11CBA", f"got {one_well['station_name']!r}"
print(one_well)
print("Correct.")

---

## Part 3 — Filtering

A comparison on a column gives a boolean Series. Indexing with it keeps the
`True` rows. This is the numpy mask from last week, with labels attached.

In [ ]:
deep = wells[wells["well_depth_ft"] > 1000]

print(f"{len(deep)} wells deeper than 1000 ft")
deep[["site_no", "station_name", "well_depth_ft"]].head()

Combine conditions with `&` (and) and `|` (or). **Each condition needs its own
parentheses** — `&` binds more tightly than `>`, so leaving them out is a
`TypeError`, and an especially unhelpful one.

In [ ]:
deep_and_high = wells[
    (wells["well_depth_ft"] > 1000) & (wells["land_surface_elev_ft"] > 2500)
]
print(f"{len(deep_and_high)} wells that are both deep and up-basin")

### YOUR TURN 3

Find the wells in the **northwest** of the study area that are **shallow**:

- latitude greater than 32.3
- longitude less than −111.0
- `well_depth_ft` less than 200

Put them in `nw_shallow`, and put the count in `n_nw_shallow`.

In [ ]:
# YOUR TURN
nw_shallow = ...
n_nw_shallow = ...

In [ ]:
# CHECK
expected = wells[
    (wells["latitude"] > 32.3)
    & (wells["longitude"] < -111.0)
    & (wells["well_depth_ft"] < 200)
]
assert n_nw_shallow == len(expected), f"expected {len(expected)}, got {n_nw_shallow}"
assert set(nw_shallow["site_no"]) == set(expected["site_no"]), "different wells selected"
print(f"{n_nw_shallow} shallow wells in the northwest.")
print(nw_shallow[["site_no", "latitude", "longitude", "well_depth_ft"]].head())
print("Correct.")

**Worth noticing:** a well missing its depth is excluded by
`well_depth_ft < 200`, because a comparison against `NaN` is always `False`.
That is usually what you want, but it is never announced. If 184 wells have no
depth on file, 184 wells silently left your analysis.

---

## Part 4 — New columns

Assigning to a column that doesn't exist creates it. The arithmetic is
elementwise, exactly like numpy.

In [ ]:
wells["well_depth_m"] = wells["well_depth_ft"] * 0.3048
wells["screen_bottom_elev_ft"] = wells["land_surface_elev_ft"] - wells["well_depth_ft"]

wells[["site_no", "land_surface_elev_ft", "well_depth_ft", "screen_bottom_elev_ft"]].head()

Now the water levels — 9,046 measurements from the 80 best-monitored wells in
the basin, going back to 1922.

In [ ]:
levels = pd.read_csv(
    DATA / "tucson_water_levels.csv",
    dtype={"site_no": str},
    parse_dates=["date"],
)

print(f"{len(levels):,} measurements, {levels['site_no'].nunique()} wells")
print(f"from {levels['date'].min():%Y-%m-%d} to {levels['date'].max():%Y-%m-%d}")
levels.head()

This is **long format**: one row per measurement, with the well identified in a
column rather than by position. Nearly all real monitoring data arrives this
way. Reshaping it is a Week 7 topic; for now we just filter it.

In [ ]:
one_site = levels[levels["site_no"] == "320824110593001"].sort_values("date")

print(f"{len(one_site)} measurements spanning "
      f"{one_site['date'].min():%Y} to {one_site['date'].max():%Y}")
one_site[["date", "depth_to_water_ft", "water_level_elev_ft"]].head()

### YOUR TURN 4

`depth_to_water_ft` counts **downward** from the land surface, so a rising
number means a falling water table — which is exactly the wrong way round for
plotting, and a reliable source of sign errors.

For `one_site`, add a column `decline_ft` giving how far the water table has
dropped **since the first measurement in the record**.

The first row should be `0.0`, and later rows positive where the level has
fallen.

In [ ]:
one_site = one_site.copy()      # avoid a SettingWithCopyWarning; see the note below

# YOUR TURN
one_site["decline_ft"] = ...

In [ ]:
# CHECK
assert "decline_ft" in one_site.columns
assert abs(one_site["decline_ft"].iloc[0]) < 1e-9, "the first measurement should be 0"
last = one_site["decline_ft"].iloc[-1]
assert abs(last - 111.05) < 0.01, f"expected about 111.05 ft of decline, got {last}"
print(f"first measurement: {one_site['date'].iloc[0]:%Y-%m-%d}, "
      f"{one_site['depth_to_water_ft'].iloc[0]:.1f} ft down")
print(f"last measurement:  {one_site['date'].iloc[-1]:%Y-%m-%d}, "
      f"{one_site['depth_to_water_ft'].iloc[-1]:.1f} ft down")
print(f"net decline:       {last:.1f} ft. Correct.")

> **That `.copy()`.** Slicing a DataFrame sometimes gives you a view and
> sometimes a copy, and assigning into a view raises `SettingWithCopyWarning`.
> When you plan to modify a subset, take an explicit `.copy()` first. It is the
> most common warning you will see this semester and the fix is always this.

A hundred and eleven feet, in one well, in one lifetime. That is what basin
groundwater depletion looks like in a table.

---

## Part 5 — The one Excel file

You will not use Excel in this course. You will absolutely be *sent* Excel
files, forever, by people who are not going to stop. Reading one is a Python
skill.

In [ ]:
perm = pd.read_excel(DATA / "week04_permeameter.xlsx", sheet_name="Constant Head")
perm.head()

`pd.read_excel` is `pd.read_csv` with a `sheet_name`. Everything after this
point is identical — which is the whole point.

### YOUR TURN 5

Compute hydraulic conductivity for each permeameter run and add it as a column
`K_m_per_day`.

For a constant-head permeameter, Darcy's law rearranges to

$$K = \frac{Q L}{A \, \Delta h}$$

where $Q$ is the flow rate, $L$ the sample length, $A$ the cross-sectional
area, and $\Delta h$ the head difference.

Two steps to get right:

- $Q$ is `volume_collected_cm3 / duration_s`, in cm³/s
- the answer comes out in cm/s; **multiply by 864 to get m/d**

In [ ]:
# YOUR TURN
area_cm2 = ...
flow_cm3_per_s = ...
perm["K_m_per_day"] = ...

In [ ]:
# CHECK
assert "K_m_per_day" in perm.columns
by_material = perm.groupby("material")["K_m_per_day"].mean()
assert abs(by_material["coarse sand"] - 28.7) < 0.5, f"got {by_material['coarse sand']:.2f}"
assert abs(by_material["fine sand"] - 4.4) < 0.2, f"got {by_material['fine sand']:.2f}"
assert abs(by_material["silty sand"] - 0.66) < 0.05, f"got {by_material['silty sand']:.2f}"
print(by_material.round(2).to_string())
print("\nThree orders of magnitude between coarse sand and silty sand. Correct.")

That spread is the single most important number in hydrogeology. Hydraulic
conductivity varies over about thirteen orders of magnitude across earth
materials — from gravel to unfractured granite — which is why a model calibrated
on the wrong lithology is not slightly wrong.

---

## Part 6 — A first plot

A DataFrame plots itself. This is not the polished version — Week 8 is about
making figures worth showing people — but it is what you use while you are
still working out what the data says.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.8))
ax.plot(one_site["date"], one_site["depth_to_water_ft"],
        color="#AB0520", lw=1.2, marker=".", ms=3)
ax.invert_yaxis()          # depth increases downward; so should the axis
ax.set_ylabel("depth to water (ft below land surface)")
ax.set_title("Well D-15-13 11CBA — one well, one century")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

Note `invert_yaxis()`. Plotting depth-below-surface the normal way up draws a
falling water table as a rising line, and someone will misread it. When the
quantity counts downward, so should the axis.

### YOUR TURN 6

Make a scatter plot of the whole well inventory: `longitude` on the x axis,
`latitude` on the y, with the marker **colour** set by `well_depth_ft`.

Store the result of `ax.scatter(...)` in `sc` so the check can look at it, and
add a colorbar.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))

# YOUR TURN
sc = ...

fig.colorbar(sc, ax=ax, label="well depth (ft)")
ax.set_xlabel("longitude")
ax.set_ylabel("latitude")
ax.set_title("Tucson basin monitoring wells")
ax.set_aspect("equal")
plt.tight_layout()
plt.show()

In [ ]:
# CHECK
assert sc is not None, "assign the result of ax.scatter(...) to sc"
assert sc.get_offsets().shape[0] == len(wells), \
    f"plotted {sc.get_offsets().shape[0]} points, expected {len(wells)}"
assert sc.get_array() is not None, "pass c=... so the colour carries the depth"
print(f"{len(wells):,} wells plotted, coloured by depth. Correct.")

**Think about this before next week:** the deep wells are not scattered at
random. They cluster, and the clusters line up with the basin axis and with
where the water has been pumped hardest. Making that legible — rather than
merely visible — is what Week 8 is about.

---

## Before you leave

1. *Kernel → Restart Kernel and Run All Cells*
2. Fix anything that breaks
3. Save

## What's due

- **HW 3 — Iterative groundwater solution**, Wednesday 9/23 at 11:59pm
- **Project 1 analysis component** — see the project brief on D2L

## Next week

Where data actually comes from: pulling records straight from the USGS API,
and dealing with the gaps, duplicates, and outliers that arrive with them.

## Stuck?

- `KeyError: 'well_depth'` means the column is spelled differently. Print
  `wells.columns` rather than guessing.
- `SettingWithCopyWarning` means you are assigning into something that might be
  a view. Take a `.copy()` when you slice.
- `TypeError: cannot compare ...` when combining filters almost always means a
  missing pair of parentheses around one of the conditions.
- A mean that comes out as `NaN` means the whole column is `NaN`; a mean that
  looks slightly off usually means some rows were skipped. `.isna().sum()`
  tells you which.
- Office hours: Tuesdays 1:00–2:00pm, Harshbarger 322B.